# **TASK1**

In [14]:
import torch
import torch.nn as nn
import random

In [15]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, emb_size, num_layers=1, dropout=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(input_size, emb_size)
        self.rnn = nn.LSTM(emb_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

In [16]:
class DecoderRNN(nn.Module):
    def __init__(self, output_size, hidden_size, emb_size, num_layers=1, dropout=0.1):
        super(DecoderRNN, self).__init__()
        self.output_size = output_size
        self.hidden_size = hidden_size
        
        self.embedding = nn.Embedding(output_size, emb_size)
        self.rnn = nn.LSTM(emb_size, hidden_size, num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.fc_out = nn.Linear(hidden_size, output_size)

    def forward(self, input, hidden, cell):
        input = input.unsqueeze(1)
        
        embedded = self.embedding(input)

        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))
        
        return prediction, hidden, cell

In [17]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device, pad_idx):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        self.pad_idx = pad_idx

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        trg_vocab_size = self.decoder.output_size

        outputs = torch.zeros(batch_size, trg_len, trg_vocab_size).to(self.device)

        hidden, cell = self.encoder(src)

        input = trg[:, 0]

        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input, hidden, cell)
            
            outputs[:, t, :] = output

            top1 = output.argmax(1)
            input = trg[:, t] if random.random() < teacher_forcing_ratio else top1

        return outputs

# Dataloading and tokenisation

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import nltk
from nltk.translate.bleu_score import corpus_bleu
import re

SOS_TOKEN = 0
EOS_TOKEN = 1
PAD_TOKEN = 2
UNK_TOKEN = 3

class Vocab:
    def __init__(self):
        self.word2index = {"<sos>": SOS_TOKEN, "<eos>": EOS_TOKEN, "<pad>": PAD_TOKEN, "<unk>": UNK_TOKEN}
        self.index2word = {SOS_TOKEN: "<sos>", EOS_TOKEN: "<eos>", PAD_TOKEN: "<pad>", UNK_TOKEN: "<unk>"}
        self.n_words = 4

    def add_sentence(self, sentence):
        for word in sentence.split():
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

def normalize_string(s):
    s = s.lower().strip()
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    return s

class TranslationDataset(Dataset):
    def __init__(self, csv_file, src_vocab=None, trg_vocab=None, build_vocab=False):
        self.df = pd.read_csv(csv_file)
        
        # NOTE: Change 'en' and 'fr' to match your exact CSV column headers
        self.src_sentences = [normalize_string(str(s)) for s in self.df['en']]
        self.trg_sentences = [normalize_string(str(s)) for s in self.df['fr']]
        
        self.src_vocab = src_vocab if src_vocab else Vocab()
        self.trg_vocab = trg_vocab if trg_vocab else Vocab()
        
        if build_vocab:
            for src, trg in zip(self.src_sentences, self.trg_sentences):
                self.src_vocab.add_sentence(src)
                self.trg_vocab.add_sentence(trg)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src_sentence = self.src_sentences[idx].split()
        trg_sentence = self.trg_sentences[idx].split()
        
        src_indices = [self.src_vocab.word2index.get(w, UNK_TOKEN) for w in src_sentence] + [EOS_TOKEN]
        trg_indices = [SOS_TOKEN] + [self.trg_vocab.word2index.get(w, UNK_TOKEN) for w in trg_sentence] + [EOS_TOKEN]
        
        return torch.tensor(src_indices), torch.tensor(trg_indices)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_padded = pad_sequence(src_batch, padding_value=PAD_TOKEN, batch_first=True)
    trg_padded = pad_sequence(trg_batch, padding_value=PAD_TOKEN, batch_first=True)
    return src_padded, trg_padded

train_data = TranslationDataset("/Users/bhuv2/Desktop/WEC Assignment/WEC_ass_new/mandatory-tasks/seq2seq-grounded-dialogue-generation/datasets from drive/en_fr_test.csv", build_vocab=True)
val_data = TranslationDataset("en_fr_val.csv", src_vocab=train_data.src_vocab, trg_vocab=train_data.trg_vocab)
test_data = TranslationDataset("en_fr_test.csv", src_vocab=train_data.src_vocab, trg_vocab=train_data.trg_vocab)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_data, batch_size=1, shuffle=False, collate_fn=collate_fn) # Batch 1 for inference

FileNotFoundError: [Errno 2] No such file or directory: 'en_fr_train.csv'

In [ ]:
import os
print("Current Working Directory:", os.getcwd())
print("Files in this directory:", os.listdir())

Current Working Directory: /Users/bhuv2/Desktop/WEC Assignment/WEC_ass_new/mandatory-tasks/seq2seq-grounded-dialogue-generation/Completed_Tasks
Files in this directory: ['TASKS.ipynb']


In [ ]:
import os

# Search for en_fr_train.csv starting from your project root
for root, dirs, files in os.walk(os.path.expanduser("~/Desktop/WEC Assignment")):
    if "en_fr_train.csv" in files:
        print("Found file at:", os.path.join(root, "en_fr_train.csv"))